# 02 — Exploratory Data Analysis
**Global Temperature Forecasting Using Multimodal Machine Learning**  
Choudhary & Kulkarni (2026), *Climatic Change* (Springer)  

Reproduces **§4** and **Figures 2–6** of the paper — 8-panel EDA:
1. Temperature anomaly time series (warm/cool shading)
2. CO₂ Keeling Curve + Sunspot Schwabe cycles
3. STL decomposition (trend, seasonal, residual)
4. Feature correlation matrix
5. ADF stationarity test results
6. Autocorrelation Function (ACF) — validates 36-month window

**Key EDA findings (paper §4):**
- ADF statistic = −2.625, p = 0.088 → non-stationary (trend-aware features needed)
- Seasonal component stationary at ±0.11°C → Fourier encoding sufficient  
- ACF peaks at lags 12 and 24, significant to lag 48 → 36-month window validated  
- CO₂ and radiative forcing perfectly collinear (r = 1.00)  
- Sunspot weakly negative with temperature (r = −0.17 to −0.20)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.tsa.seasonal import STL
from pathlib import Path

from preprocessing import load_all
from feature_engineering import build_features, get_feature_columns
from utils import set_global_seed, apply_style

set_global_seed(42)
apply_style()
FIG_DIR = Path('../results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load data
raw = load_all(use_cache=True)
df  = build_features(raw)
print(f'Feature matrix: {df.shape}')

In [ ]:
# ── Figure 2: Temperature Anomaly 1960–2024 ─────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
mask = df['date'].dt.year >= 1960
d = df[mask]
ma = d['temp_anomaly'].rolling(60, center=True).mean()

ax.fill_between(d['date'], d['temp_anomaly'], where=d['temp_anomaly']>=0,
                color='#d62728', alpha=0.65, label='Warm anomaly')
ax.fill_between(d['date'], d['temp_anomaly'], where=d['temp_anomaly']<0,
                color='#1f77b4', alpha=0.65, label='Cool anomaly')
ax.plot(d['date'], ma, 'k-', lw=2.0, label='5-yr moving average')
ax.axhline(1.5, color='#d62728', lw=1.2, ls=':', alpha=0.9, label='Paris 1.5°C')
ax.axhline(0, color='grey', lw=0.8, ls='--', alpha=0.5)
ax.set_title('Figure 2 — Global Surface Temperature Anomaly (°C), 1960–2024\n'
             'Mean warming rate +0.017°C yr⁻¹; acceleration post-2000', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Temperature Anomaly (°C)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR/'fig2_temperature_anomaly.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: CO₂ Keeling Curve + Sunspot Cycles ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
mask = df['date'].dt.year >= 1960
d = df[mask]

axes[0].plot(d['date'], d['co2_ppm'], color='#2ca02c', lw=0.9)
axes[0].set_title('CO₂ Concentration (Keeling Curve)\n315→422 ppm, gradient 0.7→2.4 ppm yr⁻¹', fontweight='bold')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('CO₂ (ppm)')

axes[1].plot(d['date'], d['sunspot_number'], color='#ff7f0e', lw=0.7, alpha=0.85)
axes[1].plot(d['date'], d['ssn_smooth11'], color='k', lw=1.8, label='11-yr Schwabe mean')
axes[1].set_title('International Sunspot Number (SILSO)\nCycles 19–25; declining amplitude Cycles 22–24', fontweight='bold')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Sunspot Number')
axes[1].legend()

plt.suptitle('Figure 3 — CO₂ and Solar Forcing Signals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'fig3_co2_sunspot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 4: STL Decomposition ─────────────────────────────────────────────
temp_series = df.set_index('date')['temp_anomaly']
stl = STL(temp_series, period=12, robust=True)
res = stl.fit()

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
axes[0].plot(temp_series.index, res.trend, color='#d62728', lw=1.5)
axes[0].set_title('Trend Component — super-linear acceleration post-2000 (0.8°C→1.5°C)', fontweight='bold')
axes[0].set_ylabel('Trend (°C)')

axes[1].plot(temp_series.index, res.seasonal, color='#1f77b4', lw=0.8, alpha=0.85)
axes[1].axhline(0.11, color='red', ls='--', lw=0.8, label='±0.11°C amplitude')
axes[1].axhline(-0.11, color='red', ls='--', lw=0.8)
axes[1].set_title('Seasonal Component — stationary at ±0.11°C (validates Fourier encoding)', fontweight='bold')
axes[1].set_ylabel('Seasonal (°C)'); axes[1].set_xlabel('Year')
axes[1].legend()

plt.suptitle('Figure 4 — STL Decomposition of Global Temperature Anomaly', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'fig4_stl_decomposition.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 5: Feature Correlation Matrix ────────────────────────────────────
selected = ['temp_anomaly','co2_ppm','radiative_forcing','sunspot_number',
            'ssn_smooth11','temp_ma60','temp_lag12','month_sin','year_norm',
            'co2_x_ssn','forcing_trend']
sel = [c for c in selected if c in df.columns]
corr = df[sel].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask_tri = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, mask=mask_tri,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Figure 5 — Feature Correlation Matrix\n'
             'co2_ppm ↔ radiative_forcing: r=1.00; sunspot ↔ temp: r=−0.17 to −0.20',
             fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'fig5_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 6: ADF Test + ACF ────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, acf as sm_acf
import matplotlib.patches as mpatches

temp = df['temp_anomaly'].dropna()
adf_result = adfuller(temp, autolag='AIC')
adf_stat, adf_p = adf_result[0], adf_result[1]
print(f'ADF Statistic: {adf_stat:.4f}  |  p-value: {adf_p:.4f}')
print('Conclusion:', 'NON-STATIONARY (p>0.05)' if adf_p > 0.05 else 'STATIONARY')

nlags = 60
acf_vals = sm_acf(temp, nlags=nlags, alpha=0.05, fft=True)
if isinstance(acf_vals, tuple):
    acf_arr, confint = acf_vals[0], acf_vals[1]
else:
    acf_arr = acf_vals; confint = None

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ADF result bar
axes[0].barh(['ADF Statistic'], [abs(adf_stat)], color='#1f77b4', alpha=0.8)
axes[0].barh(['Critical 5%'], [abs(adf_result[4]['5%'])], color='#d62728', alpha=0.6)
axes[0].set_title(f'ADF Test  (stat={adf_stat:.3f}, p={adf_p:.3f})\n'
                   f'→ p > 0.05: SERIES IS NON-STATIONARY', fontweight='bold')
axes[0].set_xlabel('|ADF Statistic|')
legend_patches = [mpatches.Patch(color='#1f77b4', label='Test stat'),
                  mpatches.Patch(color='#d62728', label='5% critical')]
axes[0].legend(handles=legend_patches)

# ACF plot
lags = np.arange(nlags + 1)
axes[1].bar(lags, acf_arr, color='#1f77b4', alpha=0.75, width=0.8)
ci = 1.96 / np.sqrt(len(temp))
axes[1].axhline(ci, color='red', ls='--', lw=0.9, label='95% CI')
axes[1].axhline(-ci, color='red', ls='--', lw=0.9)
axes[1].axvline(12, color='green', ls=':', lw=1.2, alpha=0.7, label='lag 12, 24')
axes[1].axvline(24, color='green', ls=':', lw=1.2, alpha=0.7)
axes[1].axvline(36, color='orange', ls='--', lw=1.5, alpha=0.8, label='Window=36 mo')
axes[1].set_xlabel('Lag (months)'); axes[1].set_ylabel('ACF')
axes[1].set_title('Autocorrelation Function (ACF)\nSignificant to lag 48; peaks at 12, 24 → 36-month window', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Figure 6 — ADF Stationarity Test and Autocorrelation Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'fig6_adf_acf.png', dpi=300, bbox_inches='tight')
plt.show()
print('All EDA figures saved.')